# Simple CNN testing

In [2]:
import torch
from torch.utils.data import DataLoader

import wandb

from src.cnn.cnn_models import ImprovedModel
from src.cnn.cnn_utils import get_dataset, get_device, train_eval

In [3]:
print(wandb)
print(wandb.__file__)

<module 'wandb' from '/Users/jdemid/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/wandb/__init__.py'>
/Users/jdemid/Projects/PycharmProjects/deep-learning-mpw/.venv/lib/python3.12/site-packages/wandb/__init__.py


In [4]:
# Check for GPU
device = get_device()

print(device)

mps


In [5]:
train_dataset, val_dataset = get_dataset()

In [6]:
# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

Training dataset size: 24000
Validation dataset size: 6000


### Create Training loop for models

In [ ]:
# train_eval was here

### Creating shallow CNN-model

In [ ]:
# first_model class was here

In [ ]:
# create an model and its summary

# model = first_model(128) # no need for to(device), it breaks when running on Apple mps chip
# from torchsummary import summary
# summary(model, (3,224,224),device='cpu')

Initiate Training

In [ ]:
batch_size = 64
nepochs = 50
lr = 0.001
units = 128

# model = first_model(units)
# optimizer = torch.optim.SGD(params=model.parameters(), lr = lr)
# cost_train_sgd, cost_valid_sgd, acc_train_sgd, acc_valid_sgd = train_eval(model, optimizer, nepochs, batch_size, train_dataset, val_dataset, device, entity='MSE_DeLearn_SPR26', project='MPW-CNN', run_name='CNN-testing_bs64_e30_lr0.001_u128_1l_no-reg', use_wandb=True)


This first model is heavily in a overfitting regime.\
Really bad vallidation accuracy and loss.\
The Result was kind of expected looking at the amount of parameters that are estimted during training (+40 Mio).\
Therefore we tried to construct a shallow model that performs better.
Possibilities to improve the model:
- heavier downsampling before passing into a fully connected layer
    --> adding more layers before fully connectde layers (Conv2d --> ReLu --> MaxPool2d)
- reduce/ make the dense layer smaller (less units)
- add regularization:
    - Dropout
    - better optimizer
    - early stopping
    - etc.


# Improved CNN testing

In [ ]:
# improved_model class was here

In [11]:
model = ImprovedModel() # no need for to(device), it breaks when running on Apple mps chip
from torchsummary import summary
summary(model, (3,224,224),device='cpu')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
         MaxPool2d-3         [-1, 32, 112, 112]               0
 AdaptiveAvgPool2d-4             [-1, 32, 7, 7]               0
           Flatten-5                 [-1, 1568]               0
            Linear-6                  [-1, 128]         200,832
              ReLU-7                  [-1, 128]               0
           Dropout-8                  [-1, 128]               0
            Linear-9                   [-1, 10]           1,290
Total params: 203,018
Trainable params: 203,018
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.57
Forward/backward pass size (MB): 27.59
Params size (MB): 0.77
Estimated Total Size (MB): 28.94
-----------------------------------------

In [15]:
batch_size = 64
nepochs = 5
lr = 0.01
units = 128
wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter

dl_train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dl_val = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

import src.cnn.configs as configs

train_conf = configs.TrainConfig()
train_conf.batch_size = batch_size
train_conf.nepochs = nepochs
train_conf.lr = lr
train_conf.units = units
train_conf.wd = wd

optimizer_conf = configs.OptimizerConfig()
optimizer_conf.cls = torch.optim.Adam

model_conf = configs.ModelConfig()
model_conf.cls = ImprovedModel
model_conf.kwargs = {
    "in_channels": 3,
    "num_classes": 10,
    "units": 128,
    "drop": 0.5,
}

wandb_conf = configs.WandBConfig()
wandb_conf.mode = "disabled"
experiment_conf = configs.ExperimentConfig(train=train_conf, 
                                           optimizer=optimizer_conf, 
                                           model=model_conf, 
                                           wandb=wandb_conf)

trained_model, history, result = train_eval(model,
                                            cfg=experiment_conf,
                                            train_loader=dl_train,
                                            test_loader = dl_val)


KeyboardInterrupt: 

After testing with differetn learning rates and still a huge problem with too many parameters in the model. We decided to make a small change in architecture.

In [ ]:
# improved_model class was here

In [16]:
model = ImprovedModel(128, 0.5) # no need for to(device), it breaks when running on Apple mps chip

summary(model, (3,224,224),device='cpu')

TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=NoneType), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.memory_format memory_format = None, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


By adding the AdaptiveAvgPool2d the amount of parameter is reduced from ~51 mio to 6'410.\
This setup is better suited to prevent overfitting.

In [12]:
batch_size = 64 # default batch size, can be tuned as a hyperparameter.
nepochs = 50
units = 128 # default number of units in the fully connected layer, can be tuned as a hyperparameter
wd = 1e-4 # default weight decay value for Adam optimizer, can be tuned as a hyperparameter
lrs = [0.1, 0.01, 0.001, 0.0001]

results = {}

for lr in lrs:
    print(f"\n=== Training with lr = {lr} ===")
    
    # re-initialize model every time!
    model = ImprovedModel(units, drop=0.5)
    
    # using Adam optimizer
    optimizer = torch.optim.Adam(
        params=model.parameters(),
        lr=lr,
        weight_decay=wd
    )
    
    run_name = f"CNN_bs{batch_size}_e{nepochs}_lr_{lr}_u{units}_drop0.5"
    
    cost_train, cost_valid, acc_train, acc_valid = train_eval(
        model,
        optimizer,
        nepochs,
        batch_size,
        train_dataset,
        val_dataset,
        device,
        entity='MSE_DeLearn_SPR26',
        project='MPW-CNN',
        run_name=run_name,
        use_wandb=True
    )
    
    results[lr] = {
        "train_loss": cost_train,
        "val_loss": cost_valid,
        "train_acc": acc_train,
        "val_acc": acc_valid
    }


=== Training with lr = 0.1 ===


Epoch 0: Train cost: 2.321, accuracy: 0.0992, Validation cost: 2.3178, accuracy: 0.1 (Time: 30.1 seconds)
Epoch 1: Train cost: 2.3134, accuracy: 0.0987, Validation cost: 2.3099, accuracy: 0.1 (Time: 30.9 seconds)
Epoch 2: Train cost: 2.3111, accuracy: 0.1006, Validation cost: 2.3182, accuracy: 0.1 (Time: 30.7 seconds)
Epoch 3: Train cost: 2.3139, accuracy: 0.1019, Validation cost: 2.3059, accuracy: 0.1 (Time: 31.0 seconds)
Epoch 4: Train cost: 2.3127, accuracy: 0.0998, Validation cost: 2.3233, accuracy: 0.1 (Time: 30.9 seconds)
Epoch 5: Train cost: 2.3124, accuracy: 0.0998, Validation cost: 2.3117, accuracy: 0.1 (Time: 30.9 seconds)
Epoch 6: Train cost: 2.3134, accuracy: 0.0997, Validation cost: 2.3119, accuracy: 0.1 (Time: 30.5 seconds)
Epoch 7: Train cost: 2.3149, accuracy: 0.0982, Validation cost: 2.3155, accuracy: 0.1 (Time: 30.7 seconds)
Epoch 8: Train cost: 2.3135, accuracy: 0.0987, Validation cost: 2.3097, accuracy: 0.1 (Time: 30.8 seconds)
Epoch 9: Train cost: 2.3137, accuracy:

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,▆█▇██▇▇▇▄▃▂██▄▂▁▂▁▁▂▂▂▁▂▁▁▂▂▂▂▇▂▂▂▂▂▂▁▁█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▄▃▅▇▄▄▃▃▅▂▆▅▃▄▂▄▃▄▅▂█▅▁▅▃▅▄▅▇▂▂▅▆▅▁█▂▅▁▇
train_loss,▃▂▁▂▁▂▂▂▂▅▄▁▂▅▂█▃▁▁▂▁▁▁▁▃▁▁▄▁▂▂▁▂▂▁▂▂▁▅▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▆▃▆▂█▄▅▃▅▅▄▃▁▇▄▅▁▄▄▂▄▄▆▂▄▃▂▅▃▅▂▁▄▅▄█▁▅▄▄
epoch,50
epoch_time,30.85268
lr,0.1
train_accuracy,0.10237



=== Training with lr = 0.01 ===


Epoch 0: Train cost: 2.285, accuracy: 0.1249, Validation cost: 2.2448, accuracy: 0.1587 (Time: 30.8 seconds)
Epoch 1: Train cost: 2.2364, accuracy: 0.1598, Validation cost: 2.2105, accuracy: 0.1782 (Time: 30.7 seconds)
Epoch 2: Train cost: 2.2198, accuracy: 0.1704, Validation cost: 2.1994, accuracy: 0.1838 (Time: 29.0 seconds)
Epoch 3: Train cost: 2.198, accuracy: 0.1811, Validation cost: 2.1169, accuracy: 0.2413 (Time: 28.5 seconds)
Epoch 4: Train cost: 2.1045, accuracy: 0.2307, Validation cost: 2.0534, accuracy: 0.2575 (Time: 28.4 seconds)
Epoch 5: Train cost: 2.068, accuracy: 0.2481, Validation cost: 2.0122, accuracy: 0.27 (Time: 28.8 seconds)
Epoch 6: Train cost: 2.0501, accuracy: 0.2563, Validation cost: 1.9996, accuracy: 0.2772 (Time: 30.9 seconds)
Epoch 7: Train cost: 2.0437, accuracy: 0.2578, Validation cost: 1.992, accuracy: 0.2708 (Time: 29.6 seconds)
Epoch 8: Train cost: 2.0415, accuracy: 0.2575, Validation cost: 1.9869, accuracy: 0.2795 (Time: 28.7 seconds)
Epoch 9: Train c

...

Epoch 0: Train cost: 2.2966, accuracy: 0.1165, Validation cost: 2.2801, accuracy: 0.1553 (Time: 31.0 seconds)
Epoch 1: Train cost: 2.2654, accuracy: 0.1492, Validation cost: 2.2498, accuracy: 0.1582 (Time: 31.1 seconds)
Epoch 2: Train cost: 2.2394, accuracy: 0.1658, Validation cost: 2.2298, accuracy: 0.1773 (Time: 30.3 seconds)
Epoch 3: Train cost: 2.22, accuracy: 0.1815, Validation cost: 2.2099, accuracy: 0.1875 (Time: 28.4 seconds)
Epoch 4: Train cost: 2.1923, accuracy: 0.2061, Validation cost: 2.1579, accuracy: 0.2268 (Time: 28.6 seconds)
Epoch 5: Train cost: 2.1437, accuracy: 0.225, Validation cost: 2.1073, accuracy: 0.252 (Time: 28.5 seconds)
Epoch 6: Train cost: 2.1037, accuracy: 0.2422, Validation cost: 2.0759, accuracy: 0.2595 (Time: 28.3 seconds)
Epoch 7: Train cost: 2.0787, accuracy: 0.25, Validation cost: 2.0545, accuracy: 0.268 (Time: 28.1 seconds)
Epoch 8: Train cost: 2.0622, accuracy: 0.2597, Validation cost: 2.0429, accuracy: 0.2715 (Time: 28.2 seconds)
Epoch 9: Train co

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch_time,██▆▂▃▂▁▁▁▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▂▁▁▁▂▂▂▂▂▂
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▃▃▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█████████████
train_loss,█▇▇▆▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▂▂▃▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███▇██████████
val_loss,██▇▇▆▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch,50
epoch_time,28.29033
lr,0.001
train_accuracy,0.33654



=== Training with lr = 0.0001 ===


Epoch 0: Train cost: 2.3044, accuracy: 0.1024, Validation cost: 2.3014, accuracy: 0.1295 (Time: 28.2 seconds)
Epoch 1: Train cost: 2.3014, accuracy: 0.1094, Validation cost: 2.2993, accuracy: 0.1137 (Time: 28.1 seconds)
Epoch 2: Train cost: 2.2999, accuracy: 0.1108, Validation cost: 2.2972, accuracy: 0.1337 (Time: 28.5 seconds)
Epoch 3: Train cost: 2.2968, accuracy: 0.1241, Validation cost: 2.2947, accuracy: 0.143 (Time: 28.2 seconds)
Epoch 4: Train cost: 2.2949, accuracy: 0.1267, Validation cost: 2.2916, accuracy: 0.1333 (Time: 28.1 seconds)
Epoch 5: Train cost: 2.2916, accuracy: 0.128, Validation cost: 2.2879, accuracy: 0.1387 (Time: 28.2 seconds)
Epoch 6: Train cost: 2.2869, accuracy: 0.1345, Validation cost: 2.2836, accuracy: 0.1383 (Time: 28.1 seconds)
Epoch 7: Train cost: 2.2834, accuracy: 0.1357, Validation cost: 2.2795, accuracy: 0.1395 (Time: 28.1 seconds)
Epoch 8: Train cost: 2.2786, accuracy: 0.1408, Validation cost: 2.2754, accuracy: 0.1412 (Time: 28.0 seconds)
Epoch 9: Tra

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch_time,▅▃█▄▃▃▁▁▂▅▄▄▃▂▁▃▃▁▂▃▇▅▄▄▃▅▃▇▄▆▆▄▅▄▄▅▆▆▄▄
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇███
train_loss,███▇▇▇▇▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
val_accuracy,▂▁▃▃▃▃▃▃▃▃▄▄▄▄▅▄▄▅▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
val_loss,████▇▇▇▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁
epoch,50
epoch_time,28.15384
lr,0.0001
train_accuracy,0.19925
